# Manual Feature Cleaning Review

This notebook reads `merged_feature_matrix.csv`, filters to a single instrument, inspects features in the order manual cleaning decisions should be made, and saves the cleaned modelling dataset to `{instrument}_clean_feature_set.csv`. Set `INSTRUMENT` in cell 1 to switch tickers. It does not standardise features, run CPCV, train models, or perform supervised feature selection.

## 1. Imports and paths

Set `INSTRUMENT` at the top to any ticker. The notebook reads `merged_feature_matrix.csv`, filters to that instrument, runs the feature review, and saves the cleaned dataset to `data/features/{instrument}_clean_feature_set.csv`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 160)

# â”€â”€ Change this to any instrument in the feature matrix â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
INSTRUMENT = "cl1s"
# Available: cl1s, es1s, nq1s, fesx1s, gc1s, si1s, hg1s, pl1s, ho1s, rb1s, ng1s
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "data" / "features" / "merged_feature_matrix.csv").exists():
        ROOT = candidate
        break

FEATURE_MATRIX_PATH = ROOT / "data" / "features" / "merged_feature_matrix.csv"
FEATURE_CATALOG_PATH = ROOT / "data" / "features" / "feature_catalog.csv"
CLEAN_FEATURE_SET_PATH = ROOT / "data" / "features" / f"{INSTRUMENT}_clean_feature_set.csv"

HIGH_CORR_THRESHOLD = 0.95

print(f"Instrument:    {INSTRUMENT}")
print(f"Input:         {FEATURE_MATRIX_PATH}")
print(f"Feature catalog: {FEATURE_CATALOG_PATH}")
print(f"Output:        {CLEAN_FEATURE_SET_PATH}")

Instrument:    ng1s
Input:         C:\Users\madhu\Desktop\Imperial College London\Core Modules\Systematic Trading Startegies\TestBranchAssignment\systematic-strategies-with-machine-learning-Cw\data\features\merged_feature_matrix.csv
Feature catalog: C:\Users\madhu\Desktop\Imperial College London\Core Modules\Systematic Trading Startegies\TestBranchAssignment\systematic-strategies-with-machine-learning-Cw\data\features\feature_catalog.csv
Output:        C:\Users\madhu\Desktop\Imperial College London\Core Modules\Systematic Trading Startegies\TestBranchAssignment\systematic-strategies-with-machine-learning-Cw\data\features\ng1s_clean_feature_set.csv


**After this cell:**

Paths are configured for the selected instrument. Output will be saved to `data/features/{instrument}_clean_feature_set.csv`.

## 2. Load data and filter to instrument

`merged_feature_matrix.csv` contains all instruments. We filter to the selected `INSTRUMENT` for this review.

In [2]:
full_matrix = pd.read_csv(FEATURE_MATRIX_PATH, parse_dates=["date"])
feature_catalog = pd.read_csv(FEATURE_CATALOG_PATH)
feature_group_map = feature_catalog.set_index("feature_name")["feature_group"].to_dict()

available_instruments = sorted(full_matrix["instrument"].unique())
assert INSTRUMENT in available_instruments, f"{INSTRUMENT} not found. Available: {available_instruments}"

raw = full_matrix.loc[full_matrix["instrument"].eq(INSTRUMENT)].reset_index(drop=True)
duplicate_dates = raw["date"].duplicated().sum()

print(f"Table: {INSTRUMENT} dataset summary")
summary = pd.DataFrame(
    {
        "value": [
            len(raw),
            raw.shape[1],
            raw["date"].min(),
            raw["date"].max(),
            duplicate_dates,
        ]
    },
    index=["rows", "columns", "min_date", "max_date", "duplicate_dates"],
)
display(summary)

print(f"Table: primary_signal distribution")
display(raw["primary_signal"].value_counts(dropna=False).sort_index().to_frame("count"))

print(f"Table: First rows")
display(raw.head())

Table: ng1s dataset summary


,value
rows,645
columns,190
min_date,2020-01-03 00:00:00
max_date,2022-06-30 00:00:00
duplicate_dates,0


Table: primary_signal distribution


,count
primary_signal,
-1,124
0,521


Table: First rows


,date,instrument,primary_signal,open,high,low,close,volume,open_interest,log_ret,ret_1d,ret_2d,ret_3d,ret_5d,ret_10d,ret_20d,ret_63d,ret_126d,ret_252d,mom_sign_5d,mom_sign_20d,mom_sign_63d,mom_sign_126d,mom_sign_252d,roc_5d,roc_20d,roc_63d,ret_spread_5_20,ret_spread_20_63,price_vs_sma5,price_vs_sma10,price_vs_sma20,price_vs_sma50,price_vs_sma100,price_vs_sma200,price_vs_ema5,price_vs_ema10,price_vs_ema20,price_vs_ema50,sma5_vs_sma20,sma10_vs_sma50,sma20_vs_sma50,sma50_vs_sma100,sma50_vs_sma200,sma100_vs_sma200,ema5_vs_ema20,ema12_vs_ema26,ema20_vs_ema50,ema20_vs_ema100,sma20_slope,sma50_slope,sma100_slope,ema20_slope,ema50_slope,macd_line,macd_signal,macd_hist,macd_hist_chg,macd_zscore,vol_5d,vol_10d,vol_20d,vol_63d,vol_126d,vol_252d,ewma_vol_5d,ewma_vol_10d,ewma_vol_20d,ewma_vol_63d,vol_ratio_5_20d,vol_ratio_20_63d,vol_ratio_63_126d,park_vol_5d,park_vol_10d,park_vol_20d,park_vol_63d,gk_vol_5d,gk_vol_10d,gk_vol_20d,gk_vol_63d,hl_range,co_range,intraday_ret,overnight_ret,true_range,atr_5d,atr_5d_pct,atr_10d,atr_10d_pct,atr_14d,...,stoch_d_14d,williams_r_14d,cci_14d,cci_20d,mfi_14d,uo,zscore_20d,zscore_63d,zscore_126d,bb_upper_dist,bb_lower_dist,bb_position,bb_width,bb_width_zscore,donchian_pos_20d,donchian_pos_52d,donchian_pos_252d,hmm_predicted_regime,hmm_regime_confidence,hmm_prob_low_vol,hmm_prob_normal_vol,hmm_prob_high_vol,hmm_prob_extreme_vol,hmm_prob_downside,hmm_prob_weak,hmm_prob_positive,hmm_prob_strong_upside,hmm_prob_stress,hmm_prob_upside_breakout,hmm_prob_calm_positive,hmm_prob_calm_negative,hmm_prob_chop,hmm_prob_high_or_extreme_vol,hmm_prob_low_or_normal_vol,hmm_prob_downside_or_weak,hmm_prob_positive_or_strong_upside,hmm_prob_not_stress,hmm_regime_prob_0,hmm_regime_prob_1,hmm_regime_prob_2,hmm_regime_prob_3,volume_log_change,open_interest_log_change,volume_zscore_20d,volume_zscore_63d,open_interest_zscore_20d,open_interest_zscore_63d,volume_to_open_interest,range_pct,body_pct,upper_wick_pct,lower_wick_pct,close_position_in_bar,gap_open_pct,vol_20d_for_interaction,signal_changed,days_since_signal_change,signal_rolling_mean_5d,signal_rolling_mean_20d,signal_abs_rolling_sum_20d,hmm_regime_missing,hmm_regime_0,hmm_regime_1,hmm_regime_2,hmm_regime_3,hmm_volatility_regime_code,hmm_return_regime_code,hmm_market_state_code,hmm_volatility_regime_missing,hmm_volatility_low_vol,hmm_volatility_normal_vol,hmm_volatility_high_vol,hmm_volatility_extreme_vol,hmm_return_regime_missing,hmm_return_downside,hmm_return_weak,hmm_return_positive,hmm_return_strong_upside,hmm_market_state_missing,hmm_market_stress,hmm_market_upside_breakout,hmm_market_calm_positive,hmm_market_calm_negative,hmm_market_chop,signal_x_hmm_confidence,ret_5d_x_hmm_confidence,vol_20d_x_hmm_confidence,signal_x_hmm_prob_stress,signal_x_hmm_prob_high_or_extreme_vol,signal_x_hmm_prob_positive_or_strong_upside
0,2020-01-03,ng1s,0,0.002772,0.002816,0.002712,0.002773,1.111197e+08,2.770407e+08,0.003761,0.003768,-0.026952,-0.025618,-0.067834,-0.060982,-0.105212,-0.151563,-0.182117,-0.332378,-1.0,-1.0,-1.0,-1.0,-1.0,-0.067834,-0.105212,-0.151563,0.037378,0.046351,-0.019157,-0.036741,-0.053419,-0.137404,-0.151911,-0.173773,-0.016386,-0.032508,-0.058901,-0.107865,-0.034931,-0.104502,-0.088724,-0.016818,-0.042162,-0.025778,-0.043224,-0.035988,-0.052028,-0.086990,-0.005534,-0.002577,-0.000645,-0.006162,-0.004383,-0.000108,-0.000104,-3.821122e-06,-1.592646e-06,-0.996950,0.248787,0.408976,0.398039,0.430812,0.387138,0.405435,0.340872,0.374069,0.412313,0.428813,0.625031,0.923928,1.112812,0.349985,0.316677,0.337006,0.376796,0.363484,0.319825,0.346911,0.381538,0.000104,0.000001,0.000469,0.003292,0.000104,0.000105,0.037692,0.000113,0.040745,0.000118,...,12.609667,-82.947832,-151.919741,-161.318559,20.938234,46.383604,-1.720287,-1.690328,-1.904086,0.122044,0.009176,0.069928,0.124210,-1.265828,0.130208,0.054569,0.024113,1.0,0.885336,0.111611,0.002802,0.885336,0.000251,0.885336,0.002802,0.111611,0.000251,0.885336,0.000251,0.111611,0.002802,0.0,0.885587,0.114413,0.888138,0.111862,0.1

**After this cell:**

Confirms the dataset for the selected instrument: row count, date range, signal distribution, and no duplicate dates.

## 3. Define ID, signal, and feature columns

`date` and `instrument` are identifiers. `primary_signal` is the trading signal â€” not a feature to clean. Everything else is a feature.

In [3]:
id_cols = ["date", "instrument"]
signal_col = "primary_signal"
feature_cols = [col for col in raw.columns if col not in id_cols + [signal_col]]

numeric_feature_cols = raw[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
non_numeric_feature_cols = [col for col in feature_cols if col not in numeric_feature_cols]

feature_universe = pd.DataFrame(
    {
        "count": [len(id_cols), 1, len(feature_cols), len(numeric_feature_cols), len(non_numeric_feature_cols)]
    },
    index=["id_columns", "signal_columns", "feature_columns", "numeric_feature_columns", "non_numeric_feature_columns"],
)

print("Table: Feature universe summary")
display(feature_universe)

print("Table: Feature count by catalog group")
display(
    pd.Series(feature_cols, name="feature")
    .to_frame()
    .assign(feature_group=lambda x: x["feature"].map(feature_group_map).fillna("uncatalogued"))
    .groupby("feature_group")
    .size()
    .to_frame("feature_count")
    .sort_values("feature_count", ascending=False)
)

print("Non-numeric feature columns:")
print(non_numeric_feature_cols)

Table: Feature universe summary


,count
id_columns,2
signal_columns,1
feature_columns,187
numeric_feature_columns,187
non_numeric_feature_columns,0


Table: Feature count by catalog group


,feature_count
feature_group,
technical,108
engineered_hmm_encoding,21
hmm_category_probability,18
engineered_market,14
engineered_interaction,6
hmm_latent,6
raw_market_data,6
engineered_signal,5
hmm_category_code,3


Non-numeric feature columns:
[]


**After this cell:**

This defines the full feature universe. Non-numeric features stay in the review; they are not silently removed.

## 4. Missingness inspection

Before deciding what to drop, inspect which features are missing and how concentrated the missingness is.

In [4]:
missing_pct = raw[feature_cols].isna().mean()
missing_count = raw[feature_cols].isna().sum()

missingness_table = (
    pd.DataFrame(
        {
            "feature": feature_cols,
            "feature_group": [feature_group_map.get(col, "uncatalogued") for col in feature_cols],
            "missing_count": [int(missing_count[col]) for col in feature_cols],
            "missing_pct": [missing_pct[col] for col in feature_cols],
        }
    )
    .sort_values(["missing_pct", "missing_count", "feature"], ascending=[False, False, True])
    .reset_index(drop=True)
)

print("Table: Features with missing values")
display(missingness_table.loc[missingness_table["missing_count"] > 0])

print("Table: Missingness by feature group")
display(
    missingness_table.groupby("feature_group")
    .agg(
        feature_count=("feature", "size"),
        features_with_missing=("missing_count", lambda s: int((s > 0).sum())),
        max_missing_pct=("missing_pct", "max"),
        mean_missing_pct=("missing_pct", "mean"),
    )
    .sort_values("max_missing_pct", ascending=False)
)

Table: Features with missing values


,feature,feature_group,missing_count,missing_pct
0,atr_10d,technical,17,0.026357
1,atr_10d_pct,technical,17,0.026357
2,atr_14d,technical,17,0.026357
3,atr_14d_pct,technical,17,0.026357
4,atr_20d,technical,17,0.026357
...,...,...,...,...
173,volume_zscore_63d,engineered_market,17,0.026357
174,williams_r_14d,technical,17,0.026357
175,zscore_126d,technical,17,0.026357
176,zscore_20d,technical,17,0.026357


Table: Missingness by feature group


,feature_count,features_with_missing,max_missing_pct,mean_missing_pct
feature_group,,,,
engineered_hmm_encoding,21,17,0.026357,0.021336
engineered_interaction,6,6,0.026357,0.026357
engineered_market,14,14,0.026357,0.026357
hmm_category_code,3,3,0.026357,0.026357
hmm_category_probability,18,18,0.026357,0.026357
raw_market_data,6,6,0.026357,0.026357
hmm_latent,6,6,0.026357,0.026357
technical,108,108,0.026357,0.026357
engineered_signal,5,0,0.000000,0.000000


**After this cell:**

Missingness is tiny in percentage terms, but concentrated. The next section shows the exact rows and context instead of treating missingness as a generic feature-level problem.

## 5. Perfect Duplicate Groups

Exact duplicate features are the easiest first cleaning decision. Choose one feature to keep in each group and mark the rest to drop later. The notebook suggests HMM market-state probability names where those are available, because they are the most interpretable labels.

In [5]:
normalised = raw[feature_cols].astype("string").fillna("<NA>")

parent = {feature: feature for feature in feature_cols}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    root_a = find(a)
    root_b = find(b)
    if root_a != root_b:
        parent[root_b] = root_a

seen_features = []
duplicate_feature_pairs = []
for col in feature_cols:
    duplicate_of = None
    for previous in seen_features:
        if normalised[col].equals(normalised[previous]):
            duplicate_of = previous
            union(previous, col)
            break
    if duplicate_of is not None:
        duplicate_feature_pairs.append({"feature": col, "duplicate_of": duplicate_of})
    else:
        seen_features.append(col)

duplicate_feature_pairs = pd.DataFrame(duplicate_feature_pairs, columns=["feature", "duplicate_of"])
duplicate_features = set(duplicate_feature_pairs["feature"]) if not duplicate_feature_pairs.empty else set()
duplicate_of_map = dict(zip(duplicate_feature_pairs["feature"], duplicate_feature_pairs["duplicate_of"])) if not duplicate_feature_pairs.empty else {}

clusters = {}
for feature in feature_cols:
    clusters.setdefault(find(feature), []).append(feature)
duplicate_clusters = [sorted(features) for features in clusters.values() if len(features) > 1]

preferred_duplicate_keep_order = [
    "hmm_prob_stress",
    "hmm_prob_calm_positive",
    "hmm_prob_calm_negative",
    "hmm_prob_upside_breakout",
    "hmm_regime_missing",
]

def suggest_duplicate_reason(features: list[str]) -> str:
    joined = " ".join(features)
    if "hmm_prob" in joined or "hmm_regime_prob" in joined:
        return "Exact duplicate HMM probability representation; prefer one interpretable market-state probability name."
    if "hmm_regime_missing" in joined or "_missing" in joined:
        return "Exact duplicate HMM missingness flag; keep one missing flag only."
    if any("hmm_regime_" in f for f in features) or any(f.startswith("hmm_market_") for f in features):
        return "Exact duplicate HMM one-hot/category representation; keep one representation only."
    return "Exact duplicate values across all rows; choose one feature to keep."

def suggested_keep_feature(features: list[str]) -> str:
    for preferred in preferred_duplicate_keep_order:
        if preferred in features:
            return preferred
    market_state_probs = [f for f in features if f.startswith("hmm_prob_") and f in {"hmm_prob_stress", "hmm_prob_calm_positive", "hmm_prob_calm_negative", "hmm_prob_upside_breakout"}]
    if market_state_probs:
        return sorted(market_state_probs)[0]
    return sorted(features)[0]

duplicate_group_rows = []
for group_id, features_in_group in enumerate(duplicate_clusters, start=1):
    suggested_keep = suggested_keep_feature(features_in_group)
    reason = suggest_duplicate_reason(features_in_group)
    for feature in features_in_group:
        duplicate_group_rows.append(
            {
                "duplicate_group_id": group_id,
                "feature": feature,
                "feature_group": feature_group_map.get(feature, "uncatalogued"),
                "missing_pct": missing_pct[feature],
                "unique_values": int(raw[feature].nunique(dropna=True)),
                "suggested_keep": suggested_keep,
                "suggested_reason": reason,
                "manual_keep": "",
                "manual_drop_reason": "",
            }
        )

duplicate_groups_df = pd.DataFrame(duplicate_group_rows)

print(f"Exact duplicate pairs: {len(duplicate_feature_pairs)}")
print(f"Exact duplicate groups: {len(duplicate_clusters)}")
print("Table: Exact duplicate feature pairs")
display(duplicate_feature_pairs)

for group_id in sorted(duplicate_groups_df["duplicate_group_id"].unique()):
    group_table = duplicate_groups_df.loc[duplicate_groups_df["duplicate_group_id"].eq(group_id)].copy()
    print(f"Table: Perfect duplicate group {group_id} | suggested keep = {group_table['suggested_keep'].iloc[0]}")
    display(group_table)

Exact duplicate pairs: 28
Exact duplicate groups: 10
Table: Exact duplicate feature pairs


,feature,duplicate_of
0,hmm_prob_downside,hmm_prob_high_vol
1,hmm_prob_weak,hmm_prob_normal_vol
2,hmm_prob_positive,hmm_prob_low_vol
3,hmm_prob_strong_upside,hmm_prob_extreme_vol
4,hmm_prob_stress,hmm_prob_high_vol
5,hmm_prob_upside_breakout,hmm_prob_extreme_vol
6,hmm_prob_calm_positive,hmm_prob_low_vol
7,hmm_prob_calm_negative,hmm_prob_normal_vol
8,hmm_regime_prob_0,hmm_prob_extreme_vol
9,hmm_regime_prob_1,hmm_prob_high_vol


Table: Perfect duplicate group 1 | suggested keep = hmm_prob_calm_positive


,duplicate_group_id,feature,feature_group,missing_pct,unique_values,suggested_keep,suggested_reason,manual_keep,manual_drop_reason
0,1,hmm_prob_calm_positive,hmm_category_probability,0.026357,628,hmm_prob_calm_positive,Exact duplicate HMM probability representation...,,
1,1,hmm_prob_low_vol,hmm_category_probability,0.026357,628,hmm_prob_calm_positive,Exact duplicate HMM probability representation...,,
2,1,hmm_prob_positive,hmm_category_probability,0.026357,628,hmm_prob_calm_positive,Exact duplicate HMM probability representation...,,
3,1,hmm_regime_prob_2,hmm_latent,0.026357,628,hmm_prob_calm_positive,Exact duplicate HMM probability representation...,,


Table: Perfect duplicate group 2 | suggested keep = hmm_prob_calm_negative


...

,duplicate_group_id,feature,feature_group,missing_pct,unique_values,suggested_keep,suggested_reason,manual_keep,manual_drop_reason
26,8,hmm_market_stress,engineered_hmm_encoding,0.026357,2,hmm_market_stress,Exact duplicate HMM one-hot/category represent...,,
27,8,hmm_regime_1,engineered_hmm_encoding,0.026357,2,hmm_market_stress,Exact duplicate HMM one-hot/category represent...,,
28,8,hmm_return_downside,engineered_hmm_encoding,0.026357,2,hmm_market_stress,Exact duplicate HMM one-hot/category represent...,,
29,8,hmm_volatility_high_vol,engineered_hmm_encoding,0.026357,2,hmm_market_stress,Exact duplicate HMM one-hot/category represent...,,


Table: Perfect duplicate group 9 | suggested keep = hmm_market_calm_positive


,duplicate_group_id,feature,feature_group,missing_pct,unique_values,suggested_keep,suggested_reason,manual_keep,manual_drop_reason
30,9,hmm_market_calm_positive,engineered_hmm_encoding,0.026357,2,hmm_market_calm_positive,Exact duplicate HMM one-hot/category represent...,,
31,9,hmm_regime_2,engineered_hmm_encoding,0.026357,2,hmm_market_calm_positive,Exact duplicate HMM one-hot/category represent...,,
32,9,hmm_return_positive,engineered_hmm_encoding,0.026357,2,hmm_market_calm_positive,Exact duplicate HMM one-hot/category represent...,,
33,9,hmm_volatility_low_vol,engineered_hmm_encoding,0.026357,2,hmm_market_calm_positive,Exact duplicate HMM one-hot/category represent...,,


Table: Perfect duplicate group 10 | suggested keep = hmm_market_calm_negative


,duplicate_group_id,feature,feature_group,missing_pct,unique_values,suggested_keep,suggested_reason,manual_keep,manual_drop_reason
34,10,hmm_market_calm_negative,engineered_hmm_encoding,0.026357,2,hmm_market_calm_negative,Exact duplicate HMM one-hot/category represent...,,
35,10,hmm_regime_3,engineered_hmm_encoding,0.026357,2,hmm_market_calm_negative,Exact duplicate HMM one-hot/category represent...,,
36,10,hmm_return_weak,engineered_hmm_encoding,0.026357,2,hmm_market_calm_negative,Exact duplicate HMM one-hot/category represent...,,
37,10,hmm_volatility_normal_vol,engineered_hmm_encoding,0.026357,2,hmm_market_calm_negative,Exact duplicate HMM one-hot/category represent...,,


**After this cell:**

This is the first place to make manual decisions. For each duplicate group, pick one feature to keep and record why the others should be dropped later. Nothing is dropped in this notebook.

## 6. Missing Date Investigation

The missing rows are specific calendar events, not broad random missingness. Inspect the dates before deciding whether to drop rows, add a data-quality flag, or impute.

In [6]:
missing_mask = raw[feature_cols].isna()
missing_row_counts = missing_mask.sum(axis=1)
missing_rows_df = raw.loc[missing_row_counts > 0, ["date", "instrument", "primary_signal"]].copy()
missing_rows_df["missing_feature_count"] = missing_row_counts.loc[missing_row_counts > 0].astype(int).to_numpy()
missing_rows_df["missing_feature_names"] = [
    ";".join(missing_mask.columns[row_values].tolist())
    for row_values in missing_mask.loc[missing_row_counts > 0].to_numpy()
]

# Dynamically find the window around the worst missing-value date
if len(missing_rows_df) > 0:
    worst_date = missing_rows_df.sort_values("missing_feature_count", ascending=False)["date"].iloc[0]
    context_start = worst_date - pd.Timedelta(days=4)
    context_end = worst_date + pd.Timedelta(days=4)
    context_df = raw.loc[raw["date"].between(context_start, context_end)].copy()
else:
    context_df = pd.DataFrame()

raw_context_cols = ["date", "instrument", "primary_signal", "open", "high", "low", "close", "volume", "open_interest"]
hmm_context_cols = [
    "date", "instrument", "primary_signal",
    "hmm_predicted_regime", "hmm_regime_confidence",
    "hmm_prob_stress", "hmm_prob_calm_positive", "hmm_prob_calm_negative", "hmm_prob_upside_breakout",
]
engineered_missing_context_cols = [
    "date", "instrument", "volume", "open_interest",
    "volume_log_change", "open_interest_log_change", "volume_to_open_interest",
]

print("Table: Rows with any missing feature values")
display(missing_rows_df)

if not context_df.empty:
    print(f"Table: Raw market context around worst missing date ({worst_date.date()})")
    display(context_df[[col for col in raw_context_cols if col in context_df.columns]])

    print("Table: HMM context around worst missing date")
    display(context_df[[col for col in hmm_context_cols if col in context_df.columns]])

    print("Table: Engineered volume/open-interest context around worst missing date")
    display(context_df[[col for col in engineered_missing_context_cols if col in context_df.columns]])

Table: Rows with any missing feature values


,date,instrument,primary_signal,missing_feature_count,missing_feature_names
11,2020-01-20,ng1s,0,178,open;high;low;close;volume;open_interest;log_r...
31,2020-02-17,ng1s,0,178,open;high;low;close;volume;open_interest;log_r...
100,2020-05-25,ng1s,0,178,open;high;low;close;volume;open_interest;log_r...
129,2020-07-03,ng1s,0,178,open;high;low;close;volume;open_interest;log_r...
175,2020-09-07,ng1s,0,178,open;high;low;close;volume;open_interest;log_r...
233,2020-11-26,ng1s,0,178,open;high;low;close;volume;open_interest;log_r...
268,2021-01-18,ng1s,0,178,open;high;low;close;volume;open_interest;log_r...
288,2021-02-15,ng1s,0,178,open;high;low;close;volume;open_interest;log_r...
322,2021-04-02,ng1s,0,178,open;high;low;close;volume;open_interest;log_r...
363,2021-05-31,ng1s,0,178,open;high;low;close;volume;open_interest;log_r...


Table: Raw market context around worst missing date (2020-01-20)


,date,instrument,primary_signal,open,high,low,close,volume,open_interest
9,2020-01-16,ng1s,0,0.002778,0.002825,0.002696,0.002704,1.585488e+08,1.355129e+08
10,2020-01-17,ng1s,0,0.002704,0.002705,0.002596,0.002608,1.638210e+08,1.249108e+08
11,2020-01-20,ng1s,0,NaN,NaN,NaN,NaN,NaN,NaN
12,2020-01-21,ng1s,0,0.002565,0.002565,0.002383,0.002467,1.964198e+08,1.120084e+08
13,2020-01-22,ng1s,0,0.002489,0.002528,0.002467,0.002480,1.843430e+08,6.991593e+07
14,2020-01-23,ng1s,0,0.002493,0.002578,0.002491,0.002508,1.380975e+08,4.367240e+07
15,2020-01-24,ng1s,0,0.002521,0.002527,0.002442,0.002463,1.348435e+08,4.504907e+08


Table: HMM context around worst missing date


,date,instrument,primary_signal,hmm_predicted_regime,hmm_regime_confidence,hmm_prob_stress,hmm_prob_calm_positive,hmm_prob_calm_negative,hmm_prob_upside_breakout
9,2020-01-16,ng1s,0,2.0,0.981639,1.090736e-02,9.816386e-01,0.007266,1.879395e-04
10,2020-01-17,ng1s,0,2.0,0.925731,5.077463e-02,9.257309e-01,0.022564,9.301493e-04
11,2020-01-20,ng1s,0,NaN,NaN,NaN,NaN,NaN,NaN
12,2020-01-21,ng1s,0,1.0,0.789582,7.895821e-01,3.281072e-04,0.102853,1.072369e-01
13,2020-01-22,ng1s,0,3.0,1.000000,1.277137e-11,1.601371e-08,1.000000,3.355090e-10
14,2020-01-23,ng1s,0,3.0,1.000000,6.168469e-13,1.130173e-08,1.000000,2.758736e-11
15,2020-01-24,ng1s,0,3.0,1.000000,0.000000e+00,1.482072e-295,1.000000,1.067271e-247


Table: Engineered volume/open-interest context around worst missing date


,date,instrument,volume,open_interest,volume_log_change,open_interest_log_change,volume_to_open_interest
9,2020-01-16,ng1s,1.585488e+08,1.355129e+08,-0.058803,-0.054328,1.169991
10,2020-01-17,ng1s,1.638210e+08,1.249108e+08,0.032712,-0.081466,1.311504
11,2020-01-20,ng1s,NaN,NaN,NaN,NaN,NaN
12,2020-01-21,ng1s,1.964198e+08,1.120084e+08,0.181480,-0.109027,1.753617
13,2020-01-22,ng1s,1.843430e+08,6.991593e+07,-0.063456,-0.471280,2.636638
14,2020-01-23,ng1s,1.380975e+08,4.367240e+07,-0.288838,-0.470577,3.162123
15,2020-01-24,ng1s,1.348435e+08,4.504907e+08,-0.023845,2.333621,0.299326


**After this cell:**

Missing rows are market holidays (all features NaN) and any HMM-gap dates. The context tables show the surrounding rows so you can confirm the cause before deciding to drop.

## 7. High-Correlation Deep Dive

High correlation is not automatically a drop signal. TA windows are often highly correlated because they describe related horizons. This section separates correlation pairs by type so you can inspect them more sensibly.

In [7]:
numeric_df = raw[numeric_feature_cols]
spearman_corr = numeric_df.corr(method="spearman")
notna_matrix = numeric_df.notna().astype("int8")
pairwise_counts = notna_matrix.T.dot(notna_matrix)

def family_from_feature(feature: str) -> str:
    if feature.startswith("hmm_") or "_hmm_" in feature:
        return "hmm"
    if feature in ["open", "high", "low", "close", "volume", "open_interest"]:
        return "raw_market"
    if any(token in feature for token in ["ret_", "roc_", "mom_", "slope", "sma", "ema", "macd", "rsi", "stoch", "williams", "cci", "mfi", "uo", "zscore", "bb_", "donchian", "atr", "vol_"]):
        return "technical"
    if feature in ["volume_log_change", "open_interest_log_change", "volume_zscore_20d", "volume_zscore_63d", "open_interest_zscore_20d", "open_interest_zscore_63d", "volume_to_open_interest", "range_pct", "body_pct", "upper_wick_pct", "lower_wick_pct", "close_position_in_bar", "gap_open_pct", "vol_20d_for_interaction"]:
        return "engineered_market"
    if feature.startswith("signal_") or feature.startswith("days_since_signal"):
        return "engineered_signal"
    return "other"

def correlation_bucket(row: pd.Series) -> str:
    families = {row["feature_1_family"], row["feature_2_family"]}
    if "hmm" in families:
        return "HMM duplicates/redundancies"
    if families == {"technical"}:
        return "TA window variants"
    if families == {"raw_market"}:
        return "Raw price-level correlations"
    if "technical" in families and "engineered_market" in families:
        return "Engineered-vs-technical overlaps"
    return "Other high-correlation pairs"

high_corr_rows = []
for i, feature_1 in enumerate(numeric_feature_cols):
    for feature_2 in numeric_feature_cols[i + 1:]:
        corr_value = spearman_corr.loc[feature_1, feature_2]
        if pd.isna(corr_value):
            continue
        abs_corr = abs(corr_value)
        if abs_corr >= HIGH_CORR_THRESHOLD:
            high_corr_rows.append(
                {
                    "feature_1": feature_1,
                    "feature_2": feature_2,
                    "feature_1_group": feature_group_map.get(feature_1, "uncatalogued"),
                    "feature_2_group": feature_group_map.get(feature_2, "uncatalogued"),
                    "feature_1_family": family_from_feature(feature_1),
                    "feature_2_family": family_from_feature(feature_2),
                    "spearman_corr": corr_value,
                    "abs_spearman_corr": abs_corr,
                    "n_pairwise_non_missing": int(pairwise_counts.loc[feature_1, feature_2]),
                }
            )

high_corr_pairs = (
    pd.DataFrame(high_corr_rows)
    .sort_values(["abs_spearman_corr", "n_pairwise_non_missing", "feature_1", "feature_2"], ascending=[False, False, True, True])
    .reset_index(drop=True)
    if high_corr_rows
    else pd.DataFrame(columns=["feature_1", "feature_2", "feature_1_group", "feature_2_group", "feature_1_family", "feature_2_family", "spearman_corr", "abs_spearman_corr", "n_pairwise_non_missing"])
)

if not high_corr_pairs.empty:
    high_corr_pairs["correlation_bucket"] = high_corr_pairs.apply(correlation_bucket, axis=1)
    high_corr_pairs["manual_category"] = ""
    high_corr_pairs["manual_action"] = ""
    high_corr_pairs["manual_notes"] = ""
else:
    high_corr_pairs["correlation_bucket"] = []

high_corr_features = set(high_corr_pairs["feature_1"]).union(set(high_corr_pairs["feature_2"])) if not high_corr_pairs.empty else set()
max_abs_spearman_corr = pd.Series(0.0, index=feature_cols)
if not high_corr_pairs.empty:
    stacked = pd.concat(
        [
            high_corr_pairs[["feature_1", "abs_spearman_corr"]].rename(columns={"feature_1": "feature"}),
            high_corr_pairs[["feature_2", "abs_spearman_corr"]].rename(columns={"feature_2": "feature"}),
        ],
        ignore_index=True,
    )
    max_abs_spearman_corr = stacked.groupby("feature")["abs_spearman_corr"].max().reindex(feature_cols).fillna(0.0)

print(f"High-correlation threshold: absolute Spearman >= {HIGH_CORR_THRESHOLD}")
print(f"High-correlation pairs: {len(high_corr_pairs)}")
print(f"Features in at least one high-correlation pair: {len(high_corr_features)}")
if not high_corr_pairs.empty:
    print("Table: High-correlation pair counts by review bucket")
    display(high_corr_pairs.groupby("correlation_bucket").size().to_frame("pair_count").sort_values("pair_count", ascending=False))
    for bucket_name in high_corr_pairs["correlation_bucket"].drop_duplicates().tolist():
        bucket_table = high_corr_pairs.loc[high_corr_pairs["correlation_bucket"].eq(bucket_name)].copy()
        print(f"Table: High-correlation bucket = {bucket_name} | rows = {len(bucket_table)}")
        display(bucket_table)

High-correlation threshold: absolute Spearman >= 0.95
High-correlation pairs: 170
Features in at least one high-correlation pair: 131
Table: High-correlation pair counts by review bucket


,pair_count
correlation_bucket,
TA window variants,86
HMM duplicates/redundancies,68
Other high-correlation pairs,10
Raw price-level correlations,6


Table: High-correlation bucket = HMM duplicates/redundancies | rows = 68


,feature_1,feature_2,feature_1_group,feature_2_group,feature_1_family,feature_2_family,spearman_corr,abs_spearman_corr,n_pairwise_non_missing,correlation_bucket,manual_category,manual_action,manual_notes
0,hmm_prob_calm_negative,hmm_regime_prob_3,hmm_category_probability,hmm_latent,hmm,hmm,1.000000,1.000000,116,HMM duplicates/redundancies,,,
1,hmm_prob_calm_positive,hmm_regime_prob_2,hmm_category_probability,hmm_latent,hmm,hmm,1.000000,1.000000,116,HMM duplicates/redundancies,,,
2,hmm_prob_downside,hmm_prob_stress,hmm_category_probability,hmm_category_probability,hmm,hmm,1.000000,1.000000,116,HMM duplicates/redundancies,,,
3,hmm_prob_downside,hmm_regime_prob_1,hmm_category_probability,hmm_latent,hmm,hmm,1.000000,1.000000,116,HMM duplicates/redundancies,,,
4,hmm_prob_extreme_vol,hmm_prob_strong_upside,hmm_category_probability,hmm_category_probability,hmm,hmm,1.000000,1.000000,116,HMM duplicates/redundancies,,,
5,hmm_prob_extreme_vol,hmm_prob_upside_breakout,hmm_category_probability,hmm_category_probability,hmm,hmm,1.000000,1.000000,116,HMM duplicates/redundancies,,,
6,hmm_prob_extreme_vol,hmm_regime_prob_0,hmm_category_probability,hmm_latent,hmm,hmm,1.000000,1.000000,116,HMM duplicates/redundancies,,,
7,hmm_prob_high_vol,hmm_prob_downside,hmm_category_probability,hmm_category_probability,hmm,hmm,1.000000,1.000000,116,HMM duplicates/redundancies,,,
8,hmm_prob_high_vol,hmm_prob_stress,hmm_category_probability,hmm_category_probability,hmm,hmm,1.000000,1.000000,116,HMM duplicates/redundancies,,,
9,hmm_prob_high_vol,hmm_regime_prob_1,hmm_category_probability,hmm_latent,hmm,hmm,1.000000,1.000000,116,HMM duplicates/redundancies,,,


Table: High-correlation bucket = Other high-correlation pairs | rows = 10


,feature_1,feature_2,feature_1_group,feature_2_group,feature_1_family,feature_2_family,spearman_corr,abs_spearman_corr,n_pairwise_non_missing,correlation_bucket,manual_category,manual_action,manual_notes
48,intraday_ret,body_pct,technical,engineered_market,other,engineered_market,1.000000,1.000000,116,Other high-correlation pairs,,,
49,log_ret,ret_1d,technical,technical,other,technical,1.000000,1.000000,116,Other high-correlation pairs,,,
50,overnight_ret,gap_open_pct,technical,engineered_market,other,engineered_market,1.000000,1.000000,116,Other high-correlation pairs,,,
64,signal_rolling_mean_20d,signal_abs_rolling_sum_20d,engineered_signal,engineered_signal,engineered_signal,engineered_signal,-1.000000,1.000000,-123,Other high-correlation pairs,,,
90,co_range,body_pct,technical,engineered_market,other,engineered_market,0.990596,0.990596,116,Other high-correlation pairs,,,
91,co_range,intraday_ret,technical,technical,other,other,0.990596,0.990596,116,Other high-correlation pairs,,,
96,hl_range,true_range,technical,technical,other,other,0.985883,0.985883,116,Other high-correlation pairs,,,
110,log_ret,rsi_21d_change,technical,technical,other,technical,0.975946,0.975946,116,Other high-correlation pairs,,,
117,log_ret,rsi_14d_change,technical,technical,other,technical,0.970801,0.970801,116,Other high-correlation pairs,,,
155,log_ret,rsi_7d_change,technical,technical,other,technical,0.955319,0.955319,116,Other high-correlation pairs,,,


Table: High-correlation bucket = TA window variants | rows = 86


,feature_1,feature_2,feature_1_group,feature_2_group,feature_1_family,feature_2_family,spearman_corr,abs_spearman_corr,n_pairwise_non_missing,correlation_bucket,manual_category,manual_action,manual_notes
51,price_vs_ema20,ema20_slope,technical,technical,technical,technical,1.000000,1.000000,116,TA window variants,,,
52,price_vs_ema50,ema50_slope,technical,technical,technical,technical,1.000000,1.000000,116,TA window variants,,,
53,ret_20d,roc_20d,technical,technical,technical,technical,1.000000,1.000000,116,TA window variants,,,
54,ret_5d,roc_5d,technical,technical,technical,technical,1.000000,1.000000,116,TA window variants,,,
55,ret_63d,roc_63d,technical,technical,technical,technical,1.000000,1.000000,116,TA window variants,,,
56,stoch_k_14d,williams_r_14d,technical,technical,technical,technical,1.000000,1.000000,116,TA window variants,,,
57,zscore_20d,bb_position,technical,technical,technical,technical,1.000000,1.000000,116,TA window variants,,,
71,ret_20d,sma20_slope,technical,technical,technical,technical,0.999561,0.999561,116,TA window variants,,,
72,roc_20d,sma20_slope,technical,technical,technical,technical,0.999561,0.999561,116,TA window variants,,,
73,vol_20d,vol_20d_for_interaction,technical,engineered_market,technical,technical,0.998564,0.998564,116,TA window variants,,,


Table: High-correlation bucket = Raw price-level correlations | rows = 6


,feature_1,feature_2,feature_1_group,feature_2_group,feature_1_family,feature_2_family,spearman_corr,abs_spearman_corr,n_pairwise_non_missing,correlation_bucket,manual_category,manual_action,manual_notes
79,low,close,raw_market_data,raw_market_data,raw_market,raw_market,0.995950,0.995950,116,Raw price-level correlations,,,
80,high,close,raw_market_data,raw_market_data,raw_market,raw_market,0.995815,0.995815,116,Raw price-level correlations,,,
81,open,low,raw_market_data,raw_market_data,raw_market,raw_market,0.995694,0.995694,116,Raw price-level correlations,,,
82,open,high,raw_market_data,raw_market_data,raw_market,raw_market,0.995534,0.995534,116,Raw price-level correlations,,,
86,high,low,raw_market_data,raw_market_data,raw_market,raw_market,0.994891,0.994891,116,Raw price-level correlations,,,
92,open,close,raw_market_data,raw_market_data,raw_market,raw_market,0.990280,0.990280,116,Raw price-level correlations,,,


**After this cell:**

Use the buckets differently. HMM redundancy can often be simplified aggressively. TA window variants need judgement, because nearby windows can be correlated but still represent different horizons.

## 8. Build manual feature review dataframe

`feature_review_df` is the master review surface. It includes automatic issue flags plus blank manual decision columns.

In [8]:
constant_features = set(col for col in feature_cols if raw[col].nunique(dropna=True) <= 1)

def build_issue_flags(row: pd.Series) -> str:
    flags = []
    if row["is_non_numeric"]:
        flags.append("non_numeric")
    if row["missing_pct"] > 0:
        flags.append("missing_values")
    if row["is_constant"]:
        flags.append("constant")
    if row["is_duplicate"]:
        flags.append("duplicate")
    if row["high_corr_flag"]:
        flags.append("high_corr")
    return ";".join(flags)

feature_review_df = pd.DataFrame(
    {
        "feature": feature_cols,
        "feature_group": [feature_group_map.get(col, "uncatalogued") for col in feature_cols],
        "feature_family": [family_from_feature(col) for col in feature_cols],
        "dtype": [str(raw[col].dtype) for col in feature_cols],
        "missing_pct": [missing_pct[col] for col in feature_cols],
        "unique_values": [int(raw[col].nunique(dropna=True)) for col in feature_cols],
        "is_non_numeric": [col in non_numeric_feature_cols for col in feature_cols],
        "is_constant": [col in constant_features for col in feature_cols],
        "is_duplicate": [col in duplicate_features for col in feature_cols],
        "duplicate_of": [duplicate_of_map.get(col, pd.NA) for col in feature_cols],
        "duplicate_group_id": [
            duplicate_groups_df.loc[duplicate_groups_df["feature"].eq(col), "duplicate_group_id"].iloc[0]
            if not duplicate_groups_df.empty and duplicate_groups_df["feature"].eq(col).any()
            else pd.NA
            for col in feature_cols
        ],
        "max_abs_spearman_corr": [float(max_abs_spearman_corr.get(col, 0.0)) for col in feature_cols],
        "high_corr_flag": [col in high_corr_features for col in feature_cols],
    }
)
feature_review_df["auto_issue_flags"] = feature_review_df.apply(build_issue_flags, axis=1)
feature_review_df["manual_category"] = ""
feature_review_df["manual_action"] = ""
feature_review_df["manual_notes"] = ""

feature_review_df = feature_review_df.sort_values(
    ["auto_issue_flags", "feature_group", "feature"],
    ascending=[False, True, True],
).reset_index(drop=True)

print("Table: Manual feature review dataframe")
display(feature_review_df)

print("Table: Auto issue flag counts")
display(
    feature_review_df.assign(has_issue=feature_review_df["auto_issue_flags"].ne(""))
    .groupby(["feature_group", "has_issue"])
    .size()
    .unstack(fill_value=0)
)

Table: Manual feature review dataframe


,feature,feature_group,feature_family,dtype,missing_pct,unique_values,is_non_numeric,is_constant,is_duplicate,duplicate_of,duplicate_group_id,max_abs_spearman_corr,high_corr_flag,auto_issue_flags,manual_category,manual_action,manual_notes
0,hmm_regime_0,engineered_hmm_encoding,hmm,float64,0.026357,2,False,False,False,<NA>,7,1.000000,True,missing_values;high_corr,,,
1,hmm_regime_1,engineered_hmm_encoding,hmm,float64,0.026357,2,False,False,False,<NA>,8,1.000000,True,missing_values;high_corr,,,
2,hmm_regime_2,engineered_hmm_encoding,hmm,float64,0.026357,2,False,False,False,<NA>,9,1.000000,True,missing_values;high_corr,,,
3,hmm_regime_3,engineered_hmm_encoding,hmm,float64,0.026357,2,False,False,False,<NA>,10,1.000000,True,missing_values;high_corr,,,
4,ret_5d_x_hmm_confidence,engineered_interaction,hmm,float64,0.026357,628,False,False,False,<NA>,<NA>,0.996821,True,missing_values;high_corr,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
182,hmm_return_regime_missing,engineered_hmm_encoding,hmm,int64,0.000000,2,False,False,True,hmm_regime_missing,6,1.000000,True,duplicate;high_corr,,,
183,hmm_volatility_regime_missing,engineered_hmm_encoding,hmm,int64,0.000000,2,False,False,True,hmm_regime_missing,6,1.000000,True,duplicate;high_corr,,,
184,days_since_signal_change,engineered_signal,engineered_signal,int64,0.000000,164,False,False,False,<NA>,<NA>,0.000000,False,,,,
185,signal_changed,engineered_signal,engineered_signal,int64,0.000000,2,False,False,False,<NA>,<NA>,0.000000,False,,,,


Table: Auto issue flag counts


has_issue,False,True
feature_group,,
engineered_hmm_encoding,0,21
engineered_interaction,0,6
engineered_market,0,14
engineered_signal,3,2
hmm_category_code,0,3
hmm_category_probability,0,18
hmm_latent,0,6
raw_market_data,0,6
technical,0,108


**After this cell:**

This is the table to use while categorising features manually. It does not enforce decisions; it only makes the problems visible.

## 9. Feature group review tables

These grouped views make manual inspection easier by showing related feature families together.

In [9]:
for group_name in sorted(feature_review_df["feature_group"].unique()):
    group_table = feature_review_df.loc[feature_review_df["feature_group"].eq(group_name)].copy()
    print(f"Table: Feature review group = {group_name} | rows = {len(group_table)}")
    display(group_table)

Table: Feature review group = engineered_hmm_encoding | rows = 21


,feature,feature_group,feature_family,dtype,missing_pct,unique_values,is_non_numeric,is_constant,is_duplicate,duplicate_of,duplicate_group_id,max_abs_spearman_corr,high_corr_flag,auto_issue_flags,manual_category,manual_action,manual_notes
0,hmm_regime_0,engineered_hmm_encoding,hmm,float64,0.026357,2,False,False,False,<NA>,7,1.0,True,missing_values;high_corr,,,
1,hmm_regime_1,engineered_hmm_encoding,hmm,float64,0.026357,2,False,False,False,<NA>,8,1.0,True,missing_values;high_corr,,,
2,hmm_regime_2,engineered_hmm_encoding,hmm,float64,0.026357,2,False,False,False,<NA>,9,1.0,True,missing_values;high_corr,,,
3,hmm_regime_3,engineered_hmm_encoding,hmm,float64,0.026357,2,False,False,False,<NA>,10,1.0,True,missing_values;high_corr,,,
101,hmm_market_calm_negative,engineered_hmm_encoding,hmm,float64,0.026357,2,False,False,True,hmm_regime_3,10,1.0,True,missing_values;duplicate;high_corr,,,
102,hmm_market_calm_positive,engineered_hmm_encoding,hmm,float64,0.026357,2,False,False,True,hmm_regime_2,9,1.0,True,missing_values;duplicate;high_corr,,,
103,hmm_market_stress,engineered_hmm_encoding,hmm,float64,0.026357,2,False,False,True,hmm_regime_1,8,1.0,True,missing_values;duplicate;high_corr,,,
104,hmm_market_upside_breakout,engineered_hmm_encoding,hmm,float64,0.026357,2,False,False,True,hmm_regime_0,7,1.0,True,missing_values;duplicate;high_corr,,,
105,hmm_return_downside,engineered_hmm_encoding,hmm,float64,0.026357,2,False,False,True,hmm_regime_1,8,1.0,True,missing_values;duplicate;high_corr,,,
106,hmm_return_positive,engineered_hmm_encoding,hmm,float64,0.026357,2,False,False,True,hmm_regime_2,9,1.0,True,missing_values;duplicate;high_corr,,,


Table: Feature review group = engineered_interaction | rows = 6


,feature,feature_group,feature_family,dtype,missing_pct,unique_values,is_non_numeric,is_constant,is_duplicate,duplicate_of,duplicate_group_id,max_abs_spearman_corr,high_corr_flag,auto_issue_flags,manual_category,manual_action,manual_notes
4,ret_5d_x_hmm_confidence,engineered_interaction,hmm,float64,0.026357,628,False,False,False,<NA>,<NA>,0.996821,True,missing_values;high_corr,,,
5,signal_x_hmm_confidence,engineered_interaction,hmm,float64,0.026357,116,False,False,False,<NA>,<NA>,0.995202,True,missing_values;high_corr,,,
6,signal_x_hmm_prob_high_or_extreme_vol,engineered_interaction,hmm,float64,0.026357,122,False,False,False,<NA>,<NA>,0.998245,True,missing_values;high_corr,,,
7,signal_x_hmm_prob_positive_or_strong_upside,engineered_interaction,hmm,float64,0.026357,120,False,False,False,<NA>,<NA>,0.998245,True,missing_values;high_corr,,,
8,signal_x_hmm_prob_stress,engineered_interaction,hmm,float64,0.026357,124,False,False,False,<NA>,<NA>,0.973212,True,missing_values;high_corr,,,
127,vol_20d_x_hmm_confidence,engineered_interaction,hmm,float64,0.026357,628,False,False,False,<NA>,<NA>,0.000000,False,missing_values,,,


Table: Feature review group = engineered_market | rows = 14


...

,feature,feature_group,feature_family,dtype,missing_pct,unique_values,is_non_numeric,is_constant,is_duplicate,duplicate_of,duplicate_group_id,max_abs_spearman_corr,high_corr_flag,auto_issue_flags,manual_category,manual_action,manual_notes
121,hmm_regime_prob_0,hmm_latent,hmm,float64,0.026357,619,False,False,True,hmm_prob_extreme_vol,4,1.0,True,missing_values;duplicate;high_corr,,,
122,hmm_regime_prob_1,hmm_latent,hmm,float64,0.026357,621,False,False,True,hmm_prob_high_vol,3,1.0,True,missing_values;duplicate;high_corr,,,
123,hmm_regime_prob_2,hmm_latent,hmm,float64,0.026357,628,False,False,True,hmm_prob_low_vol,1,1.0,True,missing_values;duplicate;high_corr,,,
124,hmm_regime_prob_3,hmm_latent,hmm,float64,0.026357,597,False,False,True,hmm_prob_normal_vol,2,1.0,True,missing_values;duplicate;high_corr,,,
140,hmm_predicted_regime,hmm_latent,hmm,float64,0.026357,4,False,False,False,<NA>,<NA>,0.0,False,missing_values,,,
141,hmm_regime_confidence,hmm_latent,hmm,float64,0.026357,587,False,False,False,<NA>,<NA>,0.0,False,missing_values,,,


Table: Feature review group = raw_market_data | rows = 6


,feature,feature_group,feature_family,dtype,missing_pct,unique_values,is_non_numeric,is_constant,is_duplicate,duplicate_of,duplicate_group_id,max_abs_spearman_corr,high_corr_flag,auto_issue_flags,manual_category,manual_action,manual_notes
23,close,raw_market_data,raw_market,float64,0.026357,611,False,False,False,<NA>,<NA>,0.995950,True,missing_values;high_corr,,,
24,high,raw_market_data,raw_market,float64,0.026357,618,False,False,False,<NA>,<NA>,0.995815,True,missing_values;high_corr,,,
25,low,raw_market_data,raw_market,float64,0.026357,619,False,False,False,<NA>,<NA>,0.995950,True,missing_values;high_corr,,,
26,open,raw_market_data,raw_market,float64,0.026357,617,False,False,False,<NA>,<NA>,0.995694,True,missing_values;high_corr,,,
142,open_interest,raw_market_data,raw_market,float64,0.026357,628,False,False,False,<NA>,<NA>,0.000000,False,missing_values,,,
143,volume,raw_market_data,raw_market,float64,0.026357,628,False,False,False,<NA>,<NA>,0.000000,False,missing_values,,,


Table: Feature review group = technical | rows = 108


,feature,feature_group,feature_family,dtype,missing_pct,unique_values,is_non_numeric,is_constant,is_duplicate,duplicate_of,duplicate_group_id,max_abs_spearman_corr,high_corr_flag,auto_issue_flags,manual_category,manual_action,manual_notes
27,atr_10d,technical,technical,float64,0.026357,628,False,False,False,<NA>,<NA>,0.996073,True,missing_values;high_corr,,,
28,atr_10d_pct,technical,technical,float64,0.026357,628,False,False,False,<NA>,<NA>,0.988118,True,missing_values;high_corr,,,
29,atr_14d,technical,technical,float64,0.026357,628,False,False,False,<NA>,<NA>,0.996073,True,missing_values;high_corr,,,
30,atr_14d_pct,technical,technical,float64,0.026357,628,False,False,False,<NA>,<NA>,0.988118,True,missing_values;high_corr,,,
31,atr_20d,technical,technical,float64,0.026357,628,False,False,False,<NA>,<NA>,0.995364,True,missing_values;high_corr,,,
32,atr_20d_pct,technical,technical,float64,0.026357,628,False,False,False,<NA>,<NA>,0.985302,True,missing_values;high_corr,,,
33,atr_5d,technical,technical,float64,0.026357,628,False,False,False,<NA>,<NA>,0.984287,True,missing_values;high_corr,,,
34,atr_5d_pct,technical,technical,float64,0.026357,628,False,False,False,<NA>,<NA>,0.963623,True,missing_values;high_corr,,,
35,bb_position,technical,technical,float64,0.026357,628,False,False,False,<NA>,<NA>,1.000000,True,missing_values;high_corr,,,
36,cci_14d,technical,technical,float64,0.026357,628,False,False,False,<NA>,<NA>,0.957249,True,missing_values;high_corr,,,


**After this cell:**

Reviewing by feature group is less overwhelming than scanning all features at once. Start with HMM groups and duplicate groups before worrying about TA window correlations.

## 10. Save cleaned model dataset

Applies the cleaning decisions that are obvious from the review tables: exact HMM aliases, constant HMM columns, duplicate missing flags, non-numeric labels, and simple TA overlaps. Rows with any remaining missing feature values are dropped. Output is saved to `data/features/{INSTRUMENT}_clean_feature_set.csv`.

In [10]:
hmm_constant_drop_features = ["hmm_prob_chop", "hmm_market_chop"]

hmm_missing_flag_drop_features = [
    "hmm_volatility_regime_missing",
    "hmm_return_regime_missing",
    "hmm_market_state_missing",
]

hmm_probability_alias_drop_features = [
    "hmm_prob_extreme_vol", "hmm_prob_downside", "hmm_regime_prob_0",
    "hmm_prob_normal_vol", "hmm_prob_positive", "hmm_regime_prob_1",
    "hmm_prob_low_vol", "hmm_prob_weak", "hmm_regime_prob_2",
    "hmm_prob_high_vol", "hmm_prob_strong_upside", "hmm_regime_prob_3",
    "hmm_prob_not_stress",
]

hmm_one_hot_alias_drop_features = [
    "hmm_regime_0", "hmm_regime_1", "hmm_regime_2", "hmm_regime_3",
    "hmm_volatility_extreme_vol", "hmm_volatility_normal_vol",
    "hmm_volatility_low_vol", "hmm_volatility_high_vol",
    "hmm_return_downside", "hmm_return_positive",
    "hmm_return_weak", "hmm_return_strong_upside",
]

non_numeric_hmm_label_drop_features = [
    "hmm_volatility_regime", "hmm_return_regime", "hmm_market_state",
]

hmm_category_code_drop_features = [
    "hmm_volatility_regime_code", "hmm_return_regime_code", "hmm_market_state_code",
]

simple_ta_alias_drop_features = [
    "roc_5d", "roc_20d", "roc_63d", "bb_position", "overnight_ret", "intraday_ret",
]

return_ladder_drop_features = ["log_ret", "ret_2d", "ret_3d", "ret_10d"]

hard_drop_features = (
    hmm_constant_drop_features
    + hmm_missing_flag_drop_features
    + hmm_probability_alias_drop_features
    + hmm_one_hot_alias_drop_features
    + non_numeric_hmm_label_drop_features
    + hmm_category_code_drop_features
    + simple_ta_alias_drop_features
    + return_ladder_drop_features
)
hard_drop_features = [f for f in hard_drop_features if f in raw.columns]

# Drop rows that still have any missing feature values after column cleaning
clean = raw.drop(columns=hard_drop_features).copy()
model_feature_cols = [col for col in clean.columns if col not in id_cols + [signal_col]]
missing_after_drop = clean[model_feature_cols].isna().any(axis=1)
clean = clean.loc[~missing_after_drop].reset_index(drop=True)

hard_drop_summary = pd.DataFrame(
    {
        "drop_group": (
            ["constant_hmm"] * len(hmm_constant_drop_features)
            + ["duplicate_missing_flag"] * len(hmm_missing_flag_drop_features)
            + ["hmm_probability_alias"] * len(hmm_probability_alias_drop_features)
            + ["hmm_one_hot_alias"] * len(hmm_one_hot_alias_drop_features)
            + ["non_numeric_hmm_label"] * len(non_numeric_hmm_label_drop_features)
            + ["hmm_category_code"] * len(hmm_category_code_drop_features)
            + ["simple_ta_alias"] * len(simple_ta_alias_drop_features)
            + ["return_ladder_trim"] * len(return_ladder_drop_features)
        ),
        "feature": (
            hmm_constant_drop_features + hmm_missing_flag_drop_features
            + hmm_probability_alias_drop_features + hmm_one_hot_alias_drop_features
            + non_numeric_hmm_label_drop_features + hmm_category_code_drop_features
            + simple_ta_alias_drop_features + return_ladder_drop_features
        ),
    }
).loc[lambda df: df["feature"].isin(hard_drop_features)].reset_index(drop=True)

model_dataset_summary = pd.DataFrame(
    {
        "metric": ["original_rows", "clean_rows", "rows_removed",
                   "original_columns", "clean_columns", "features_removed", "remaining_feature_columns"],
        "value": [
            len(raw), len(clean), len(raw) - len(clean),
            raw.shape[1], clean.shape[1], len(hard_drop_features), len(model_feature_cols),
        ],
    }
)

print("Table: Hard-drop feature list")
display(hard_drop_summary)

print("Table: Before/after summary")
display(model_dataset_summary)

print("Table: Clean dataset sample")
display(clean.head())

clean.to_csv(CLEAN_FEATURE_SET_PATH, index=False)
print(f"\nSaved: {CLEAN_FEATURE_SET_PATH}")
print(f"Rows: {clean.shape[0]:,} | Columns: {clean.shape[1]:,}")

Table: Hard-drop feature list


,drop_group,feature
0,constant_hmm,hmm_prob_chop
1,constant_hmm,hmm_market_chop
2,duplicate_missing_flag,hmm_volatility_regime_missing
3,duplicate_missing_flag,hmm_return_regime_missing
4,duplicate_missing_flag,hmm_market_state_missing
5,hmm_probability_alias,hmm_prob_extreme_vol
6,hmm_probability_alias,hmm_prob_downside
7,hmm_probability_alias,hmm_regime_prob_0
8,hmm_probability_alias,hmm_prob_normal_vol
9,hmm_probability_alias,hmm_prob_positive


Table: Before/after summary


,metric,value
0,original_rows,645
1,clean_rows,628
2,rows_removed,17
3,original_columns,190
4,clean_columns,147
5,features_removed,43
6,remaining_feature_columns,144


Table: Clean dataset sample


,date,instrument,primary_signal,open,high,low,close,volume,open_interest,ret_1d,ret_5d,ret_20d,ret_63d,ret_126d,ret_252d,mom_sign_5d,mom_sign_20d,mom_sign_63d,mom_sign_126d,mom_sign_252d,ret_spread_5_20,ret_spread_20_63,price_vs_sma5,price_vs_sma10,price_vs_sma20,price_vs_sma50,price_vs_sma100,price_vs_sma200,price_vs_ema5,price_vs_ema10,price_vs_ema20,price_vs_ema50,sma5_vs_sma20,sma10_vs_sma50,sma20_vs_sma50,sma50_vs_sma100,sma50_vs_sma200,sma100_vs_sma200,ema5_vs_ema20,ema12_vs_ema26,ema20_vs_ema50,ema20_vs_ema100,sma20_slope,sma50_slope,sma100_slope,ema20_slope,ema50_slope,macd_line,macd_signal,macd_hist,macd_hist_chg,macd_zscore,vol_5d,vol_10d,vol_20d,vol_63d,vol_126d,vol_252d,ewma_vol_5d,ewma_vol_10d,ewma_vol_20d,ewma_vol_63d,vol_ratio_5_20d,vol_ratio_20_63d,vol_ratio_63_126d,park_vol_5d,park_vol_10d,park_vol_20d,park_vol_63d,gk_vol_5d,gk_vol_10d,gk_vol_20d,gk_vol_63d,hl_range,co_range,true_range,atr_5d,atr_5d_pct,atr_10d,atr_10d_pct,atr_14d,atr_14d_pct,atr_20d,atr_20d_pct,rsi_7d,rsi_7d_change,rsi_14d,rsi_14d_change,rsi_21d,rsi_21d_change,stoch_k_14d,stoch_d_14d,williams_r_14d,cci_14d,cci_20d,mfi_14d,uo,zscore_20d,zscore_63d,zscore_126d,bb_upper_dist,bb_lower_dist,bb_width,bb_width_zscore,donchian_pos_20d,donchian_pos_52d,donchian_pos_252d,hmm_predicted_regime,hmm_regime_confidence,hmm_prob_stress,hmm_prob_upside_breakout,hmm_prob_calm_positive,hmm_prob_calm_negative,hmm_prob_high_or_extreme_vol,hmm_prob_low_or_normal_vol,hmm_prob_downside_or_weak,hmm_prob_positive_or_strong_upside,volume_log_change,open_interest_log_change,volume_zscore_20d,volume_zscore_63d,open_interest_zscore_20d,open_interest_zscore_63d,volume_to_open_interest,range_pct,body_pct,upper_wick_pct,lower_wick_pct,close_position_in_bar,gap_open_pct,vol_20d_for_interaction,signal_changed,days_since_signal_change,signal_rolling_mean_5d,signal_rolling_mean_20d,signal_abs_rolling_sum_20d,hmm_regime_missing,hmm_market_stress,hmm_market_upside_breakout,hmm_market_calm_positive,hmm_market_calm_negative,signal_x_hmm_confidence,ret_5d_x_hmm_confidence,vol_20d_x_hmm_confidence,signal_x_hmm_prob_stress,signal_x_hmm_prob_high_or_extreme_vol,signal_x_hmm_prob_positive_or_strong_upside
0,2020-01-03,ng1s,0,0.002772,0.002816,0.002712,0.002773,1.111197e+08,2.770407e+08,0.003768,-0.067834,-0.105212,-0.151563,-0.182117,-0.332378,-1.0,-1.0,-1.0,-1.0,-1.0,0.037378,0.046351,-0.019157,-0.036741,-0.053419,-0.137404,-0.151911,-0.173773,-0.016386,-0.032508,-0.058901,-0.107865,-0.034931,-0.104502,-0.088724,-0.016818,-0.042162,-0.025778,-0.043224,-0.035988,-0.052028,-0.086990,-0.005534,-0.002577,-0.000645,-0.006162,-0.004383,-0.000108,-0.000104,-3.821122e-06,-1.592646e-06,-0.996950,0.248787,0.408976,0.398039,0.430812,0.387138,0.405435,0.340872,0.374069,0.412313,0.428813,0.625031,0.923928,1.112812,0.349985,0.316677,0.337006,0.376796,0.363484,0.319825,0.346911,0.381538,0.000104,0.000001,0.000104,0.000105,0.037692,0.000113,0.040745,0.000118,0.042482,0.000122,0.043951,32.932620,1.781543,36.890701,0.722540,38.993770,0.446578,17.052168,12.609667,-82.947832,-151.919741,-161.318559,20.938234,46.383604,-1.720287,-1.690328,-1.904086,0.122044,0.009176,0.124210,-1.265828,0.130208,0.054569,0.024113,1.0,0.885336,0.885336,0.000251,0.111611,0.002802,0.885587,0.114413,0.888138,0.111862,-0.128888,-0.011761,-0.697344,-0.982784,0.781867,0.961560,0.401095,0.037557,0.000469,0.015492,0.021597,0.587518,0.003298,0.025030,0,0,0.0,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,-0.060056,0.022160,0.0,0.0,0.0
1,2020-01-06,ng1s,0,0.002750,0.002829,0.002733,0.002780,1.188075e+08,2.770199e+08,0.002348,-0.043032,-0.113459,-0.157886,-0.175080,-0.352576,-1.0,-1.0,-1.0,-1.0,-1.0,0.070427,0.044427,-0.008084,-0.029193,-0.045401,-0.133098,-0.149234,-0.170316,-0.009429,-0.024876,-0.051571,-0.102046,-0.037621,-0.107030,-0.091868,-0.018613,-0.042932,-0.024780,-0.042543,-0.036261,-0.053220,-0.089417,-0.006071,-0.002631,-0.000808,-0.005399,-0.004148,-0.000108,-0.000105,-3.257338e-06,5.637844e-07,-0.974756,0.252994,0.411289,0.394081,0.430193,0


Saved: C:\Users\madhu\Desktop\Imperial College London\Core Modules\Systematic Trading Startegies\TestBranchAssignment\systematic-strategies-with-machine-learning-Cw\data\features\ng1s_clean_feature_set.csv


Rows: 628 | Columns: 147


**After this cell:**

The clean dataset is saved to `data/features/{instrument}_clean_feature_set.csv`. Duplicate/alias features are removed and rows with missing values are dropped. High-correlation TA families that are not simple aliases remain for later CPCV-safe selection.

## 11. Validation

Confirms the clean dataset was saved correctly: no missing values, no dropped features remaining, signal column present, and output file exists.

In [11]:
assert INSTRUMENT in full_matrix["instrument"].values, f"{INSTRUMENT} not found in feature matrix."
assert len(raw) > 0, "No rows after filtering to instrument."
assert duplicate_dates == 0, "Duplicate dates found for this instrument."
assert len(feature_review_df) == len(feature_cols), "feature_review_df should have one row per feature column."
assert set(feature_review_df["feature"]) == set(feature_cols), "feature_review_df feature set does not match feature columns."
assert "feature_review_df" in globals(), "feature_review_df was not created."
assert "high_corr_pairs" in globals(), "high_corr_pairs was not created."
assert "duplicate_feature_pairs" in globals(), "duplicate_feature_pairs was not created."
assert "duplicate_groups_df" in globals(), "duplicate_groups_df was not created."
assert "clean" in globals(), "clean dataset was not created."
assert CLEAN_FEATURE_SET_PATH.exists(), f"Output file was not saved: {CLEAN_FEATURE_SET_PATH}"
assert not set(hard_drop_features).intersection(clean.columns), "Hard-drop features still present in clean dataset."
assert clean[model_feature_cols].isna().sum().sum() == 0, "Clean dataset still has missing feature values."
assert signal_col in clean.columns, "primary_signal column is missing from clean dataset."

print(f"Validation passed.")
print(f"Instrument : {INSTRUMENT}")
print(f"Clean rows : {len(clean):,}")
print(f"Features   : {len(model_feature_cols)}")
print(f"Saved to   : {CLEAN_FEATURE_SET_PATH}")

Validation passed.


Instrument : ng1s
Clean rows : 628
Features   : 144
Saved to   : C:\Users\madhu\Desktop\Imperial College London\Core Modules\Systematic Trading Startegies\TestBranchAssignment\systematic-strategies-with-machine-learning-Cw\data\features\ng1s_clean_feature_set.csv


**After this cell:**

All checks passed. The clean feature set for the selected instrument has been saved and is ready for modelling.